# Canada Trade Data Pipeline

Processes Canada's import/export trade data from Statistics Canada (Feb 2021 - Feb 2026) using medallion architecture.

**Source**: [StatsCan Trade Data](https://www150.statcan.gc.ca/n1/pub/71-607-x/71-607-x2021004-eng.htm) - manually downloaded and imported to Google Sheets
- Selected commodity within StatsCan Data: "2709.00.10 - Petroleum & bituminous min oils, crude, relative density >= 0.0942 (<25° A.P.I.)"

**Pipeline**:
1. Load data from Google Sheets → clean column names
2. Convert dates → write to bronze tables
3. Drop unused columns, filter zero quantities → write to gold tables
4. Verify table creation and data quality

**Output Tables**:
* `bronze.ca_trade_export` / `bronze.ca_trade_import` - Raw data with date conversion
* `gold.ca_trade_export_cleaned` / `gold.ca_trade_import_cleaned` - Cleaned, analysis-ready data

**Schema**: Period (date), Province, Country, Value_$, Quantity


In [0]:
import pandas as pd
import numpy as np

In [0]:
# Data was obtained from StatsCan using their webapp to download import and export data for canada for the last 5 years (Feb 2021 - Feb 2026)
# https://www150.statcan.gc.ca/n1/pub/71-607-x/71-607-x2021004-eng.htm

# Imports - https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=0#gid=0
# Exports - https://docs.google.com/spreadsheets/d/1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y/edit?gid=1809920598#gid=1809920598

sheet_id = "1LYwuRewKYRpudNzhzupYvEyNScuHxG3amPnpgJC5z6Y"
import_gid = "0" # sheet tab ID
export_gid = "1809920598"
import_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={import_gid}"
export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={export_gid}"

import_df = spark.createDataFrame(pd.read_csv(import_url))
export_df = spark.createDataFrame(pd.read_csv(export_url))

# Clean column names 
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

import_df = clean_column_names(import_df)
export_df = clean_column_names(export_df)

In [0]:
#change Period column to datetime
from pyspark.sql.functions import to_date, col

import_df = import_df.withColumn("Period", to_date(col("Period"), "yyyy-MM-dd"))
export_df = export_df.withColumn("Period", to_date(col("Period"), "yyyy-MM-dd"))

#write to bronze table
export_df.write.mode("overwrite").format("delta").saveAsTable("bronze.CA_trade_export")
import_df.write.mode("overwrite").format("delta").saveAsTable("bronze.CA_trade_import")

#check dtypes of columns
export_df.dtypes
import_df.dtypes

[('Period', 'date'),
 ('Commodity', 'string'),
 ('Province', 'string'),
 ('Country', 'string'),
 ('State', 'string'),
 ('Value_$', 'double'),
 ('Quantity', 'bigint'),
 ('Unit_of_measure', 'string')]

In [0]:
# display(import_df)

In [0]:
# drop columns that are not needed for analysis
import_df = import_df.drop("Unit_of_measure", "State", "Commodity")
export_df = export_df.drop("Unit_of_measure", "State", "Commodity")

#check for any entries that has "Quantity" = 0 or N/A
import_df = import_df.filter(import_df.Quantity != 0)
export_df = export_df.filter(export_df.Quantity != 0)

In [0]:
#write to gold table
export_df.write.mode("overwrite").format("delta").saveAsTable(
    "gold.CA_trade_export_cleaned"
)
import_df.write.mode("overwrite").format("delta").saveAsTable(
    "gold.CA_trade_import_cleaned"
)

In [0]:
#verify table
df_sample = spark.table("gold.CA_trade_export_cleaned")
print(f"Total rows: {df_sample.count()}")
display(df_sample)

Total rows: 1291


Period,Province,Country,Value_$,Quantity
2026-05-01,Canada,United States,2.17E8,265075
2026-05-01,Canada,United States,4.89E8,644072
2021-06-01,Canada,United States,5.6420797E7,106039
2021-07-01,Canada,United States,9.1173679E7,171465
2021-06-01,Canada,United States,4.5948154E7,99013
2021-06-01,Canada,United States,3.4410229E7,69521
2021-06-01,Canada,United States,7.5E8,1714271
2021-07-01,Canada,Singapore,4059396.0,8585
2021-08-01,Canada,United States,7.4756024E7,144101
2021-09-01,Canada,United States,2.2500292E7,51963


In [0]:
#verify table
df_sample = spark.table("gold.CA_trade_import_cleaned")
print(f"Total rows: {df_sample.count()}")
display(df_sample)

Total rows: 91


Period,Province,Country,Value_$,Quantity
2022-01-01,Canada,United States,5.0344837E7,85591
2022-07-01,Canada,Ecuador,9.1920352E7,108306
2023-04-01,Canada,United States,1.25E8,186993
2023-09-01,Canada,United States,1.37E8,191192
2024-01-01,Canada,United States,5.7857829E7,99469
2024-07-01,Canada,United States,9.6123157E7,140740
2021-11-01,Canada,Colombia,5.5246458E7,86938
2022-02-01,Canada,Colombia,6.3483876E7,90479
2022-03-01,Canada,United States,4.4584824E7,58984
2022-04-01,Canada,Colombia,8.4150976E7,95143
